
# GVH Diagonal Cubic 0.3.2.7.3.7.3.3.1.6
## Strict Full-Field Canonical and Dirac HH Residual Classification

**Auteur :** Charlemagne O Laurince  
**Branche :** `0.2C1_prediction_foundations`  
**Prédécesseurs directs :** `.1.3`, `.1.4`, `.1.5`  
**Nature :** gate décisif de suffisance fonctionnelle et tentative de matérialisation de \(R_{HH}\)  
**Traceabilité :** `ACTUAL-GVH-UPSTREAM / NO-FORCED-CLOSURE / NO-FABRICATION`

---

# Question

Construire, si et seulement si toutes les dérivées fonctionnelles nécessaires sont effectivement disponibles,

\[
R_{HH}^{\rm can}[N,M]
=
\{H[N],H[M]\}_{\rm can}
-
D[\beta],
\]

avec

\[
\beta^i=h^{ij}(ND_jM-MD_jN),
\]

puis le résidu réduit par crochet de Dirac

\[
R_{HH}^{D}[N,M]
=
\{H[N],H[M]\}_{D}
-
D[\beta].
\]

Trois classifications physiques seulement sont autorisées :

\[
R_{HH}=0,
\qquad
R_{HH}\approx0,
\qquad
R_{HH}\neq0\ \text{irréductible}.
\]

Mais aucune de ces trois étiquettes ne sera produite si un bloc fonctionnel full-field manque encore.

\[
\boxed{\mathrm{DISPERSION\_READY=False}}
\]

à l'entrée.


In [1]:

from __future__ import annotations
import sympy as sp
import json, sys
from pathlib import Path

print("GVH 0.3.2.7.3.7.3.3.1.6")
print("Python:", sys.version.split()[0])
print("SymPy:", sp.__version__)


GVH 0.3.2.7.3.7.3.3.1.6
Python: 3.12.13
SymPy: 1.14.0



# 1 — Contrat canonique exact

Les secteurs canoniques hérités sont :

\[
(h_{ij},\pi^{ij}),
\qquad
(s,p_s),
\qquad
(v_i,p_v^{\,i}),
\]

avec normalisation métrique symétrique :

\[
\boxed{
\{h_{ij}(x),\pi^{kl}(y)\}
=
\frac12
(\delta_i^k\delta_j^l+\delta_i^l\delta_j^k)
\delta^3(x-y).
}
\]

La contrainte de momentum full-field déjà dérivée est :

\[
\boxed{
\mathcal C_i
=
-2D_j\pi^j{}_i
+
p_sD_is
+
p_v^jD_iv_j
-
D_j(p_v^jv_i).
}
\]

Donc :

\[
D[\beta]
=
\int d^3x\,\beta^i\mathcal C_i.
\]


In [2]:

def kd(a,b):
    return sp.Integer(1) if a == b else sp.Integer(0)

def sym_delta(i,j,k,l):
    return sp.Rational(1,2)*(
        kd(i,k)*kd(j,l)
        +
        kd(i,l)*kd(j,k)
    )

assert sym_delta(0,1,0,1) == sp.Rational(1,2)
assert sym_delta(0,1,1,0) == sp.Rational(1,2)
assert sym_delta(0,0,0,0) == 1

STRICT_METRIC_SYMMETRIZATION = True

CI_EXACT = (
    "-2 D_j pi^j_i + p_s D_i s "
    "+ p_v^j D_i v_j - D_j(p_v^j v_i)"
)

print("STRICT_METRIC_SYMMETRIZATION =",STRICT_METRIC_SYMMETRIZATION)
print("C_i =",CI_EXACT)


STRICT_METRIC_SYMMETRIZATION = True
C_i = -2 D_j pi^j_i + p_s D_i s + p_v^j D_i v_j - D_j(p_v^j v_i)



# 2 — Reconstruction du secteur \(Q,J,U\)

On reprend exactement la représentation locale amont :

\[
V^A=
(K_{11},K_{22},K_{33},K_{12},K_{13},K_{23},S,W_1,W_2,W_3),
\]

\[
A=-S-v^ia_i,
\]

\[
B_i=s\,a_i+W_i-K_i{}^jv_j,
\]

\[
C_i=-D_is-K_i{}^jv_j,
\]

\[
D_{ij}=D_iv_j+sK_{ij}.
\]

Le but est de vérifier que le noyau compact réellement utilisé par la chaîne est disponible ici aussi.


In [3]:

c1,c2,c3,c4,s = sp.symbols("c1 c2 c3 c4 s", real=True)

v = sp.Matrix(sp.symbols("v1:4", real=True))
avec = sp.Matrix(sp.symbols("a1:4", real=True))
Gs = sp.Matrix(sp.symbols("g1:4", real=True))

qsyms = sp.symbols(
    "q11 q12 q13 q21 q22 q23 q31 q32 q33",
    real=True
)
Qv = sp.Matrix(3,3,qsyms)

K11,K22,K33,K12,K13,K23,Sdot,Vdot1,Vdot2,Vdot3 = sp.symbols(
    "K11 K22 K33 K12 K13 K23 Sdot Vdot1 Vdot2 Vdot3",
    real=True
)

vel = sp.Matrix([
    K11,K22,K33,K12,K13,K23,
    Sdot,Vdot1,Vdot2,Vdot3
])

K = sp.Matrix([
    [K11,K12,K13],
    [K12,K22,K23],
    [K13,K23,K33]
])

Vdot = sp.Matrix([Vdot1,Vdot2,Vdot3])

A = sp.expand(-Sdot-v.dot(avec))
Bvec = sp.expand(s*avec+Vdot-K*v)
Cvec = sp.expand(-Gs-K*v)
Dmat = sp.expand(Qv+s*K)

I1 = sp.expand(
    A**2
    - Bvec.dot(Bvec)
    - Cvec.dot(Cvec)
    + sum(Dmat[i,j]**2 for i in range(3) for j in range(3))
)

theta = sp.expand(-A+sp.trace(Dmat))

I3 = sp.expand(
    A**2
    - 2*Bvec.dot(Cvec)
    + sum(Dmat[i,j]*Dmat[j,i] for i in range(3) for j in range(3))
)

alpha = sp.expand(s*A+v.dot(Cvec))
beta_vec = sp.expand(s*Bvec+Dmat.T*v)
acc2 = sp.expand(-alpha**2+beta_vec.dot(beta_vec))

Lu = sp.expand(
    -c1*I1
    -c2*theta**2
    -c3*I3
    +c4*acc2
)

LEH = sp.expand(
    sum(K[i,j]**2 for i in range(3) for j in range(3))
    - sp.trace(K)**2
)

zero_vel = {x:0 for x in vel}

Qu = sp.hessian(Lu,list(vel))

QEH = sp.zeros(10,10)
QEH6 = sp.hessian(LEH,list(vel[:6]))

for i in range(6):
    for j in range(6):
        QEH[i,j] = QEH6[i,j]

Q = sp.simplify(Qu+QEH)

J = sp.Matrix([
    sp.simplify(sp.diff(Lu,x).subs(zero_vel))
    for x in vel
])

U = sp.simplify(Lu.subs(zero_vel))
L = J.jacobian(avec)
G = sp.hessian(U,list(avec))

assert Q == Q.T
assert all(not Q.has(a) for a in avec)

QJU_RECONSTRUCTED = True

print("Q shape =",Q.shape)
print("J shape =",J.shape)
print("Q/J/U reconstruction: PASS")


Q shape = (10, 10)
J shape = (10, 1)
Q/J/U reconstruction: PASS



# 3 — Identités de null-direction et \(B^i\)

Les identités doivent rester :

\[
QU_{\rm shift}+L=0,
\qquad
L^TU_{\rm shift}+G=0.
\]

Elles donnent :

\[
-L^TQ^{-1}=U_{\rm shift}^T
\]

sur la branche inversible et, après `.1.4`,

\[
\boxed{
B^i
=
-\frac{v^ip_s+s\,p_v^i}{\sqrt h}.
}
\]


In [4]:

Ushift = sp.zeros(10,3)

for i in range(3):
    Ushift[6,i] = -v[i]
    Ushift[7+i,i] = -s

assert sp.simplify(Q*Ushift+L) == sp.zeros(10,3)
assert sp.simplify(L.T*Ushift+G) == sp.zeros(3,3)

zero_a = {a:0 for a in avec}

J0 = sp.Matrix([
    sp.simplify(x.subs(zero_a))
    for x in J
])

U0 = sp.simplify(U.subs(zero_a))

uvec = sp.Matrix([
    sp.simplify(sp.diff(U,a).subs(zero_a))
    for a in avec
])

P = sp.Matrix(sp.symbols("P0:10", real=True))

B_local = sp.Matrix([
    sp.simplify(x)
    for x in (
        Ushift.T*(P-J0)-uvec
    )
])

B_expected = sp.Matrix([
    -v[0]*P[6]-s*P[7],
    -v[1]*P[6]-s*P[8],
    -v[2]*P[6]-s*P[9],
])

assert sp.simplify(B_local-B_expected) == sp.zeros(3,1)

NULL_DIRECTION_IDENTITIES = True
B_FULL_FIELD_FROM_1_4 = True

print("null-direction identities: PASS")
print("B_local:")
sp.pprint(B_local)


null-direction identities: PASS
B_local:
⎡-P₆⋅v₁ - P₇⋅s⎤
⎢             ⎥
⎢-P₆⋅v₂ - P₈⋅s⎥
⎢             ⎥
⎣-P₆⋅v₃ - P₉⋅s⎦



# 4 — Bloc \(D_iB^i\) : dérivées déjà fermées

Avec :

\[
I_B[N]
=
\int d^3x\sqrt h\,ND_iB^i
=
\int d^3x\,
(v^ip_s+s\,p_v^i)D_iN
\]

à un terme de bord près, `.1.4` a fourni :

\[
\frac{\delta I_B}{\delta h_{kl}}
=
-\frac{p_s}{2}
(v^kD^lN+v^lD^kN),
\]

\[
\frac{\delta I_B}{\delta\pi^{kl}}=0,
\]

\[
\frac{\delta I_B}{\delta s}=p_v^iD_iN,
\qquad
\frac{\delta I_B}{\delta p_s}=v^iD_iN,
\]

\[
\frac{\delta I_B}{\delta v_j}=p_sD^jN,
\qquad
\frac{\delta I_B}{\delta p_v^j}=sD_jN.
\]

On enregistre ce bloc comme fermé.


In [5]:

DIVB_FUNCTIONAL_BLOCKS = {
    "delta_h":
        "-(p_s/2)*(v^k D^l N + v^l D^k N)",
    "delta_pi":
        "0",
    "delta_s":
        "p_v^i D_i N",
    "delta_p_s":
        "v^i D_i N",
    "delta_v_j":
        "p_s D^j N",
    "delta_p_v_j":
        "s D_j N",
}

DIVB_FULL_GVH_ALL_PAIRS_EXPANDED = True

assert len(DIVB_FUNCTIONAL_BLOCKS) == 6

print(
    "DIVB_FULL_GVH_ALL_PAIRS_EXPANDED =",
    DIVB_FULL_GVH_ALL_PAIRS_EXPANDED
)


DIVB_FULL_GVH_ALL_PAIRS_EXPANDED = True



# 5 — Bloc de courbure spatiale

Le bloc redérivé dans `.1.3` est :

\[
\boxed{
\frac{\delta}{\delta h_{ij}}
\int d^3x\,N\sqrt h\,R^{(3)}
=
\sqrt h
\left[
-NG^{ij}
+
D^iD^jN
-
h^{ij}D^2N
\right]
}
\]

à termes de bord près.

Le présent notebook ne remplace pas ce résultat par un proxy discret.


In [6]:

Nsym, sqrt_h = sp.symbols("N sqrt_h")
Gij, HessNij, hij_inv, D2N = sp.symbols(
    "Gij HessNij hInvij D2N"
)

R3_EULER = sqrt_h*(
    -Nsym*Gij
    +HessNij
    -hij_inv*D2N
)

R3_FUNCTIONAL_DERIVATIVE_AVAILABLE = True

print("R3 Euler tensor template =",R3_EULER)


R3 Euler tensor template = sqrt_h*(-D2N*hInvij - Gij*N + HessNij)



# 6 — Secteur auxiliaire réel et crochet de Dirac

`.1.5` a établi sur la branche générique :

\[
\Phi_A
=
(p_\lambda,\chi,\psi,\rho),
\]

\[
\det C_4=\Delta^4,
\qquad
\operatorname{rank}C_4=4.
\]

Le secteur auxiliaire est donc second-class et la classification finale doit distinguer :

\[
R_{HH}^{\rm can}
\]

de :

\[
R_{HH}^{D}.
\]


In [7]:

c14, ct = sp.symbols(
    "c14 c_time",
    nonzero=True,
    real=True
)

sa,v1a,v2a,v3a = sp.symbols(
    "s_a v1_a v2_a v3_a",
    real=True
)

psa,p1a,p2a,p3a = sp.symbols(
    "ps_a p1_a p2_a p3_a",
    real=True
)

lam,plam = sp.symbols(
    "lambda_mult p_lambda",
    real=True
)

v2a = v1a**2+v2a**2+v3a**2
p2a = p1a**2+p2a**2+p3a**2
vpa = v1a*p1a+v2a*p2a+v3a*p3a

chi = sp.expand(-sa**2+v2a+1)

psi = sp.factor(
    sa*psa/ct
    +
    vpa/c14
)

Delta = sp.factor(
    2*(v2a/c14-sa**2/ct)
)

Akin = sp.factor(
    -psa**2/(2*ct**2)
    +
    p2a/(2*c14**2)
)

rho = sp.factor(
    Akin+lam*Delta
)

xaux,yaux = sp.symbols("x_aux y_aux")

C4_struct = sp.Matrix([
    [0,0,0,-Delta],
    [0,0,Delta,xaux],
    [0,-Delta,0,yaux],
    [Delta,-xaux,-yaux,0],
])

C4_inv = sp.Matrix([
    [0,yaux/Delta**2,-xaux/Delta**2,1/Delta],
    [-yaux/Delta**2,0,-1/Delta,0],
    [xaux/Delta**2,1/Delta,0,0],
    [-1/Delta,0,0,0],
])

assert sp.simplify(
    C4_struct*C4_inv-sp.eye(4)
) == sp.zeros(4)

assert sp.factor(C4_struct.det()-Delta**4) == 0

AUXILIARY_BASIS_ACTUAL = True
DIRAC_MATRIX_READY = True

print("det(C4)=Delta^4: PASS")
print("Dirac inverse template: PASS")


det(C4)=Delta^4: PASS
Dirac inverse template: PASS



# 7 — Normalisation des moments métriques collectifs

Avant tout HH strict, il faut éviter un facteur deux caché.

La base locale emploie :

\[
(K_{11},K_{22},K_{33},K_{12},K_{13},K_{23}).
\]

Mais la 1-forme canonique tensorielle est :

\[
\pi^{ij}\dot h_{ij}
=
\pi^{11}\dot h_{11}
+\pi^{22}\dot h_{22}
+\pi^{33}\dot h_{33}
+
2\pi^{12}\dot h_{12}
+
2\pi^{13}\dot h_{13}
+
2\pi^{23}\dot h_{23}.
\]

Avec :

\[
K_{ij}
=
\frac{1}{2N}
(\dot h_{ij}-\mathcal L_{\vec N}h_{ij}),
\]

on obtient :

\[
\boxed{
P_{ii}
=
\frac{2\pi^{ii}}{\sqrt h},
\qquad
P_{ij}
=
\frac{4\pi^{ij}}{\sqrt h}
\quad(i<j).
}
\]

Cette étape dérive explicitement le lift collectif \(\to\) densité canonique.


In [8]:

N0, sh = sp.symbols(
    "N0 sqrt_h",
    nonzero=True
)

P11,P22,P33,P12,P13,P23 = sp.symbols(
    "P11 P22 P33 P12 P13 P23"
)

dh11,dh22,dh33,dh12,dh13,dh23 = sp.symbols(
    "dh11 dh22 dh33 dh12 dh13 dh23"
)

pi11,pi22,pi33,pi12,pi13,pi23 = sp.symbols(
    "pi11 pi22 pi33 pi12 pi13 pi23"
)

# Coefficients cinématiques de N sqrt(h) P_A K_A,
# avec K_A = dot(h_A)/(2N) pour les variables symétriques indépendantes.
legendre_linear = sh*sp.Rational(1,2)*(
    P11*dh11
    +P22*dh22
    +P33*dh33
    +P12*dh12
    +P13*dh13
    +P23*dh23
)

canonical_linear = (
    pi11*dh11
    +pi22*dh22
    +pi33*dh33
    +2*pi12*dh12
    +2*pi13*dh13
    +2*pi23*dh23
)

metric_map = {
    P11:2*pi11/sh,
    P22:2*pi22/sh,
    P33:2*pi33/sh,
    P12:4*pi12/sh,
    P13:4*pi13/sh,
    P23:4*pi23/sh,
}

mapped = sp.expand(
    legendre_linear.subs(metric_map)
)

assert sp.expand(
    mapped-canonical_linear
) == 0

METRIC_COLLECTIVE_TO_CANONICAL_DENSITY_MAP = True

print(
    "METRIC_COLLECTIVE_TO_CANONICAL_DENSITY_MAP =",
    METRIC_COLLECTIVE_TO_CANONICAL_DENSITY_MAP
)


METRIC_COLLECTIVE_TO_CANONICAL_DENSITY_MAP = True



# 8 — Dérivée exacte du secteur Legendre sans développer \(Q^{-1}\)

À \(a_i=0\), la contrainte locale est :

\[
C_N^{\rm loc}
=
-F_0,
\]

\[
F_0
=
\frac12(P-J_0)^TQ^{-1}(P-J_0)-U_0.
\]

Posons :

\[
X
=
Q^{-1}(P-J_0).
\]

Alors :

\[
\boxed{
\frac{\partial C_N^{\rm loc}}{\partial P_A}
=
-X_A
}
\]

et, pour toute variable locale \(z\),

\[
\boxed{
\frac{\partial C_N^{\rm loc}}{\partial z}
=
J_{0,z}^TX
+
\frac12X^TQ_{,z}X
+
U_{0,z}.
}
\]

Cette identité permet de dériver les jets locaux sans afficher un \(Q^{-1}\) gigantesque.


In [9]:

# Contrôle algébrique indépendant sur une matrice 2x2 dépendant de z.
z = sp.symbols("z", real=True)
pA,pB = sp.symbols("pA pB", real=True)

Qt = sp.Matrix([
    [2+z,1],
    [1,3]
])

Jt = sp.Matrix([
    z,
    z**2
])

Ut = z**3
Pt = sp.Matrix([pA,pB])

rt = Pt-Jt
Xt = sp.simplify(Qt.inv()*rt)

Ct = sp.simplify(
    -sp.Rational(1,2)*(rt.T*Qt.inv()*rt)[0]
    +Ut
)

dC_direct = sp.simplify(
    sp.diff(Ct,z)
)

dC_identity = sp.simplify(
    (sp.diff(Jt,z).T*Xt)[0]
    +
    sp.Rational(1,2)*
    (Xt.T*sp.diff(Qt,z)*Xt)[0]
    +
    sp.diff(Ut,z)
)

assert sp.simplify(
    dC_direct-dC_identity
) == 0

for k,pvar in enumerate([pA,pB]):
    assert sp.simplify(
        sp.diff(Ct,pvar)+Xt[k]
    ) == 0

CNLOC_LEGENDRE_DERIVATIVE_IDENTITY = True

print(
    "CNLOC_LEGENDRE_DERIVATIVE_IDENTITY =",
    CNLOC_LEGENDRE_DERIVATIVE_IDENTITY
)


CNLOC_LEGENDRE_DERIVATIVE_IDENTITY = True



# 9 — Le secteur local dépend réellement des gradients spatiaux

Pour un Hamiltonien local :

\[
C(q,p,Dq),
\]

la partie du crochet HH contenant les dérivées des smearings dépend notamment de :

\[
\frac{\partial C}{\partial(D_iq^A)}
\frac{\partial C}{\partial p_A}.
\]

Il faut donc vérifier que les objets :

\[
D_is,\qquad D_iv_j
\]

ne sont pas fictifs dans \(J_0,U_0\).

S'ils apparaissent réellement, leurs variations full-field doivent être conservées.


In [10]:

gradient_symbols = list(Gs)+list(Qv)

J_GRADIENT_DEPENDENCE_COUNT = sum(
    1
    for expr in J0
    for g in gradient_symbols
    if sp.diff(expr,g) != 0
)

U_GRADIENT_DEPENDENCE_COUNT = sum(
    1
    for g in gradient_symbols
    if sp.diff(U0,g) != 0
)

CNLOC_SPATIAL_GRADIENT_DEPENDENCE_PRESENT = (
    J_GRADIENT_DEPENDENCE_COUNT > 0
    or
    U_GRADIENT_DEPENDENCE_COUNT > 0
)

assert CNLOC_SPATIAL_GRADIENT_DEPENDENCE_PRESENT

print(
    "J gradient-dependence count =",
    J_GRADIENT_DEPENDENCE_COUNT
)
print(
    "U gradient-dependence count =",
    U_GRADIENT_DEPENDENCE_COUNT
)
print(
    "CNLOC_SPATIAL_GRADIENT_DEPENDENCE_PRESENT =",
    CNLOC_SPATIAL_GRADIENT_DEPENDENCE_PRESENT
)


J gradient-dependence count = 51
U gradient-dependence count = 12
CNLOC_SPATIAL_GRADIENT_DEPENDENCE_PRESENT = True



# 10 — Gate covariant métrique du secteur \(C_N^{\rm loc}\)

Voici le point décisif.

Dans la représentation locale utilisée pour \(Q,J,U\),

\[
q_{ij}\equiv D_iv_j
\]

est représenté par neuf symboles indépendants.

Mais full-field :

\[
D_iv_j
=
\partial_iv_j-\Gamma^k{}_{ij}v_k,
\]

donc :

\[
\boxed{
\delta(D_iv_j)
=
D_i(\delta v_j)
-
\delta\Gamma^k{}_{ij}\,v_k
}
\]

et :

\[
\boxed{
\delta\Gamma^k{}_{ij}
=
\frac12h^{kl}
\left(
D_i\delta h_{jl}
+
D_j\delta h_{il}
-
D_l\delta h_{ij}
\right).
}
\]

Ainsi la variation métrique de \(C_N^{\rm loc}\) contient, en général, des termes issus de la connexion.

En plus :

\[
v^i=h^{ij}v_j
\]

apparaît dans le secteur local, donc ses contractions ont aussi une dépendance métrique.

Le code local orthonormé \(Q,J,U\) ne contient ni \(h_{ij}\) ni \(\Gamma^k{}_{ij}\). Il ne peut donc **pas à lui seul** produire le véritable opérateur :

\[
\frac{\delta}{\delta h_{ij}}
\int N C_N^{\rm loc}.
\]

Cette absence doit être traitée comme un gate matériel, et non remplacée par l'hypothèse \(\delta(D_iv_j)/\delta h_{kl}=0\).


In [11]:

# Démonstration structurelle minimale :
# si C dépend de Qv_ij = D_i v_j, une variation de connexion donne
# delta C = (dC/dQv_ij) * (-deltaGamma^k_ij v_k) + ...

A_Qv, deltaGamma, v_k = sp.symbols(
    "A_Qv deltaGamma v_k",
    nonzero=True
)

connection_metric_variation_term = sp.expand(
    -A_Qv*deltaGamma*v_k
)

assert connection_metric_variation_term != 0

# La dépendance réelle de J0/U0 à Qv a déjà été testée.
Qv_dependence_present = (
    any(
        sp.diff(expr,q) != 0
        for expr in J0
        for q in list(Qv)
    )
    or
    any(
        sp.diff(U0,q) != 0
        for q in list(Qv)
    )
)

assert Qv_dependence_present

CNLOC_QV_COVARIANT_METRIC_RESPONSE_REQUIRED = True

# Ce notebook n'a pas encore une expression covariante tensorielle de
# C_N^loc avec h_ij, h^ij, Gamma^k_ij et leurs jets explicitement présents.
CNLOC_COVARIANT_METRIC_FUNCTIONAL_JET_EXPLICIT = False

print(
    "Qv dependence present =",
    Qv_dependence_present
)
print(
    "connection metric-variation term structurally nonzero =",
    connection_metric_variation_term
)
print(
    "CNLOC_COVARIANT_METRIC_FUNCTIONAL_JET_EXPLICIT =",
    CNLOC_COVARIANT_METRIC_FUNCTIONAL_JET_EXPLICIT
)


Qv dependence present = True
connection metric-variation term structurally nonzero = -A_Qv*deltaGamma*v_k
CNLOC_COVARIANT_METRIC_FUNCTIONAL_JET_EXPLICIT = False



# 11 — Pourquoi ce gate est indispensable pour \(HH\)

Le crochet strict contient :

\[
\int d^3x
\left[
\frac{\delta H[N]}{\delta h_{ij}}
\frac{\delta H[M]}{\delta\pi^{ij}}
-(N\leftrightarrow M)
\right].
\]

La dérivée :

\[
\frac{\delta H}{\delta\pi^{ij}}
\]

peut être obtenue à partir de l'identité de Legendre et du mapping collectif \(\leftrightarrow\pi^{ij}\).

Mais si :

\[
\frac{\delta H}{\delta h_{ij}}
\]

omet la variation métrique de :

\[
D_iv_j,\quad v^i,\quad
\text{et les contractions contenues dans }Q,J,U,
\]

alors le coefficient de :

\[
ND_iM-MD_iN
\]

est incomplet.

On ne peut donc pas comparer honnêtement le résultat à :

\[
D[\beta].
\]

Ce notebook refuse en particulier la substitution non justifiée :

\[
\boxed{
\text{« local orthonormal }Q,J,U\text{ »}
\Rightarrow
\text{« variation métrique full-field complète »}.
}
\]


In [12]:

PREREQUISITES = {
    "strict_metric_symmetrization":
        STRICT_METRIC_SYMMETRIZATION,

    "exact_Ci_available":
        True,

    "QJU_reconstructed":
        QJU_RECONSTRUCTED,

    "null_direction_identities":
        NULL_DIRECTION_IDENTITIES,

    "B_full_field_all_pairs":
        B_FULL_FIELD_FROM_1_4,

    "DivB_all_pair_functional_derivatives":
        DIVB_FULL_GVH_ALL_PAIRS_EXPANDED,

    "R3_metric_functional_derivative":
        R3_FUNCTIONAL_DERIVATIVE_AVAILABLE,

    "actual_auxiliary_basis":
        AUXILIARY_BASIS_ACTUAL,

    "Dirac_matrix_ready":
        DIRAC_MATRIX_READY,

    "metric_collective_to_pi_density_map":
        METRIC_COLLECTIVE_TO_CANONICAL_DENSITY_MAP,

    "CNloc_Legendre_derivative_identity":
        CNLOC_LEGENDRE_DERIVATIVE_IDENTITY,

    "CNloc_spatial_gradient_dependence_present":
        CNLOC_SPATIAL_GRADIENT_DEPENDENCE_PRESENT,

    "CNloc_covariant_metric_functional_jet_explicit":
        CNLOC_COVARIANT_METRIC_FUNCTIONAL_JET_EXPLICIT,
}

for k,vv in PREREQUISITES.items():
    print(k,":",vv)


strict_metric_symmetrization : True
exact_Ci_available : True
QJU_reconstructed : True
null_direction_identities : True
B_full_field_all_pairs : True
DivB_all_pair_functional_derivatives : True
R3_metric_functional_derivative : True
actual_auxiliary_basis : True
Dirac_matrix_ready : True
metric_collective_to_pi_density_map : True
CNloc_Legendre_derivative_identity : True
CNloc_spatial_gradient_dependence_present : True
CNloc_covariant_metric_functional_jet_explicit : False



# 12 — Gate de matérialisation de \(R_{HH}^{\rm can}\)

La règle est :

\[
\boxed{
\texttt{FULL\_HH\_CANONICAL\_BRACKET\_COMPUTED=True}
}
\]

uniquement si **tous** les blocs fonctionnels full-field sont réellement présents.

Les blocs déjà fermés sont :

- \(R^{(3)}\) ;
- \(D_iB^i\) ;
- base auxiliaire et réduction ;
- inverse de Dirac ;
- normalisation des moments métriques.

Le bloc restant découvert par l'assemblage final est :

\[
\boxed{
\frac{\delta}{\delta h_{ij}}
\int N C_N^{\rm loc}
}
\]

dans sa représentation covariante complète, incluant la variation de \(D_iv_j\) via la connexion.

Ce gate n'était pas équivalent au bloc \(R^{(3)}\) ni au bloc \(D_iB^i\).


In [13]:

FULL_FUNCTIONAL_JET_COMPLETE = all(
    PREREQUISITES.values()
)

if FULL_FUNCTIONAL_JET_COMPLETE:
    FULL_HH_CANONICAL_BRACKET_COMPUTED = True
    RHH_CANONICAL_MATERIALIZED = True
    RHH_CANONICAL_STATUS = (
        "READY-FOR-EXACT-RESIDUAL-CLASSIFICATION"
    )
else:
    FULL_HH_CANONICAL_BRACKET_COMPUTED = False
    RHH_CANONICAL_MATERIALIZED = False
    RHH_CANONICAL_STATUS = (
        "BLOCKED-MISSING-CNLOC-COVARIANT-METRIC-FUNCTIONAL-JET"
    )

assert not FULL_FUNCTIONAL_JET_COMPLETE
assert not FULL_HH_CANONICAL_BRACKET_COMPUTED
assert not RHH_CANONICAL_MATERIALIZED

print(
    "FULL_FUNCTIONAL_JET_COMPLETE =",
    FULL_FUNCTIONAL_JET_COMPLETE
)
print(
    "FULL_HH_CANONICAL_BRACKET_COMPUTED =",
    FULL_HH_CANONICAL_BRACKET_COMPUTED
)
print(
    "RHH_CANONICAL_STATUS =",
    RHH_CANONICAL_STATUS
)


FULL_FUNCTIONAL_JET_COMPLETE = False
FULL_HH_CANONICAL_BRACKET_COMPUTED = False
RHH_CANONICAL_STATUS = BLOCKED-MISSING-CNLOC-COVARIANT-METRIC-FUNCTIONAL-JET



# 13 — Gate du crochet de Dirac

Le crochet réduit exige :

\[
\{H[N],H[M]\}_D
=
\{H[N],H[M]\}_{\rm can}
+
a_A[N](C^{-1})^{AB}a_B[M],
\]

où :

\[
a_A[N]
=
\{H[N],\Phi_A\}_{\rm can}.
\]

L'inverse \(C^{-1}\) est disponible grâce à `.1.5`.

Mais la correction de Dirac ne peut pas remplacer un crochet canonique non matérialisé.

Donc :

\[
\boxed{
R_{HH}^{D}
\text{ reste lui aussi non matérialisé}
}
\]

tant que le gate covariant de \(C_N^{\rm loc}\) est ouvert.


In [14]:

if FULL_HH_CANONICAL_BRACKET_COMPUTED:
    FULL_HH_DIRAC_BRACKET_COMPUTED = True
    RHH_DIRAC_STATUS = (
        "READY-FOR-DIRAC-CORRECTION"
    )
else:
    FULL_HH_DIRAC_BRACKET_COMPUTED = False
    RHH_DIRAC_STATUS = (
        "BLOCKED-BY-CANONICAL-HH-NOT-MATERIALIZED"
    )

assert not FULL_HH_DIRAC_BRACKET_COMPUTED

print(
    "FULL_HH_DIRAC_BRACKET_COMPUTED =",
    FULL_HH_DIRAC_BRACKET_COMPUTED
)
print(
    "RHH_DIRAC_STATUS =",
    RHH_DIRAC_STATUS
)


FULL_HH_DIRAC_BRACKET_COMPUTED = False
RHH_DIRAC_STATUS = BLOCKED-BY-CANONICAL-HH-NOT-MATERIALIZED



# 14 — Classification physique : interdiction de faux verdict

À ce stade, il serait incorrect d'écrire :

\[
R_{HH}=0,
\]

ou :

\[
R_{HH}\approx0,
\]

ou même :

\[
R_{HH}\neq0.
\]

Pourquoi ?

Parce que :

\[
R_{HH}
\]

n'a pas encore été entièrement matérialisé.

Le verdict scientifique correct est donc un **blocage de classification**, pas une fermeture ni une déformation.


In [15]:

RHH_PHYSICAL_CLASSIFIED = False
HYPERSURFACE_ALGEBRA_CLOSED = False
DISPERSION_READY = False

R_HH_STATUS = (
    "BLOCKED-MISSING-CNLOC-COVARIANT-METRIC-FUNCTIONAL-JET"
)

assert not RHH_PHYSICAL_CLASSIFIED
assert not HYPERSURFACE_ALGEBRA_CLOSED
assert not DISPERSION_READY

print(
    "RHH_PHYSICAL_CLASSIFIED =",
    RHH_PHYSICAL_CLASSIFIED
)
print(
    "HYPERSURFACE_ALGEBRA_CLOSED =",
    HYPERSURFACE_ALGEBRA_CLOSED
)
print(
    "R_HH_STATUS =",
    R_HH_STATUS
)
print(
    "DISPERSION_READY =",
    DISPERSION_READY
)


RHH_PHYSICAL_CLASSIFIED = False
HYPERSURFACE_ALGEBRA_CLOSED = False
R_HH_STATUS = BLOCKED-MISSING-CNLOC-COVARIANT-METRIC-FUNCTIONAL-JET
DISPERSION_READY = False



# 15 — Ce que `.1.6` a néanmoins fermé

Cette étape apporte deux résultats supplémentaires réels :

### A. normalisation métrique collective

\[
\boxed{
P_{ii}=\frac{2\pi^{ii}}{\sqrt h},
\qquad
P_{ij}=\frac{4\pi^{ij}}{\sqrt h}\;(i<j)
}
\]

avec vérification de la 1-forme canonique.

### B. identité différentielle du Legendre compact

\[
\boxed{
\partial_z C_N^{\rm loc}
=
J_{0,z}^TX
+
\frac12X^TQ_{,z}X
+
U_{0,z}
}
\]

avec :

\[
X=Q^{-1}(P-J_0).
\]

Cela réduit fortement le prochain calcul : il n'est pas nécessaire de développer brutalement tout \(Q^{-1}\).

Le seul bloc à covariantiser est désormais clairement identifié.


In [16]:

NEW_CLOSURES = {
    "metric_collective_momentum_density_map":
        METRIC_COLLECTIVE_TO_CANONICAL_DENSITY_MAP,

    "CNloc_compact_Legendre_derivative_identity":
        CNLOC_LEGENDRE_DERIVATIVE_IDENTITY,

    "missing_covariant_metric_jet_identified":
        True,
}

assert all(NEW_CLOSURES.values())

for k,vv in NEW_CLOSURES.items():
    print(k,":",vv)


metric_collective_momentum_density_map : True
CNloc_compact_Legendre_derivative_identity : True
missing_covariant_metric_jet_identified : True



# 16 — Prochaine sous-étape autorisée

Le bloc manquant est suffisamment précis pour justifier une seule sous-étape ciblée :

## `0.3.2.7.3.7.3.3.1.6.1 — Covariant CNloc Metric Functional Jet and Connection-Variation Closure`

Elle devra partir du **même** \(C_N^{\rm loc}\), sans changer le modèle, et construire explicitement :

\[
\frac{\delta}{\delta h_{ij}}
\int d^3x\,N C_N^{\rm loc}
\]

en gardant :

\[
\delta h^{ij},
\]

\[
\delta v^i,
\]

\[
\delta(D_iv_j)
=
D_i\delta v_j-\delta\Gamma^k{}_{ij}v_k,
\]

et le mapping :

\[
P_{ii}=\frac{2\pi^{ii}}{\sqrt h},
\qquad
P_{ij}=\frac{4\pi^{ij}}{\sqrt h}.
\]

Une fois ce bloc fermé, il faudra revenir directement à l'assemblage HH, sans ajouter d'autre mécanisme physique.

Aucun PPN, aucune dispersion et aucune branche RACC ne doivent intervenir ici.


In [17]:

GATES = {
    "QJU_reconstructed":
        QJU_RECONSTRUCTED,

    "strict_metric_symmetrization":
        STRICT_METRIC_SYMMETRIZATION,

    "exact_Ci_available":
        True,

    "R3_functional_derivative_available":
        R3_FUNCTIONAL_DERIVATIVE_AVAILABLE,

    "DivB_full_GVH_all_pairs_expanded":
        DIVB_FULL_GVH_ALL_PAIRS_EXPANDED,

    "actual_auxiliary_basis_ready":
        AUXILIARY_BASIS_ACTUAL,

    "Dirac_inverse_ready":
        DIRAC_MATRIX_READY,

    "metric_collective_to_pi_density_map":
        METRIC_COLLECTIVE_TO_CANONICAL_DENSITY_MAP,

    "CNloc_Legendre_derivative_identity":
        CNLOC_LEGENDRE_DERIVATIVE_IDENTITY,

    "CNloc_spatial_gradient_dependence_present":
        CNLOC_SPATIAL_GRADIENT_DEPENDENCE_PRESENT,

    "CNloc_covariant_metric_functional_jet_explicit":
        CNLOC_COVARIANT_METRIC_FUNCTIONAL_JET_EXPLICIT,

    "full_HH_canonical_bracket_computed":
        FULL_HH_CANONICAL_BRACKET_COMPUTED,

    "full_HH_Dirac_bracket_computed":
        FULL_HH_DIRAC_BRACKET_COMPUTED,

    "RHH_physical_classified":
        RHH_PHYSICAL_CLASSIFIED,

    "hypersurface_algebra_closed":
        HYPERSURFACE_ALGEBRA_CLOSED,

    "dispersion_ready":
        DISPERSION_READY,
}

for k,vv in GATES.items():
    print(k,":",vv)


QJU_reconstructed : True
strict_metric_symmetrization : True
exact_Ci_available : True
R3_functional_derivative_available : True
DivB_full_GVH_all_pairs_expanded : True
actual_auxiliary_basis_ready : True
Dirac_inverse_ready : True
metric_collective_to_pi_density_map : True
CNloc_Legendre_derivative_identity : True
CNloc_spatial_gradient_dependence_present : True
CNloc_covariant_metric_functional_jet_explicit : False
full_HH_canonical_bracket_computed : False
full_HH_Dirac_bracket_computed : False
RHH_physical_classified : False
hypersurface_algebra_closed : False
dispersion_ready : False



# 17 — Verdict de `.1.6`

Le gate décisif ne doit pas être considéré comme « raté » : il a testé si la chaîne était réellement suffisante pour une classification HH.

Le résultat méthodologique autorisé est :

\[
\boxed{
\texttt{
PARTIAL-PASS-HH-PREREQUISITE-ASSEMBLY-
METRIC-MOMENTUM-LIFT-AND-LEGENDRE-DERIVATIVE-CLOSED
}
}
\]

mais :

\[
\boxed{
\texttt{
BLOCKED-CNLOC-COVARIANT-METRIC-FUNCTIONAL-JET
}
}
\]

donc :

\[
\boxed{
R_{HH}=\texttt{NOT-YET-MATERIALIZED}
}
\]

et non :

\[
R_{HH}=0,\quad
R_{HH}\approx0,\quad
R_{HH}\neq0.
\]

La distinction est essentielle : **aucune conclusion physique sur la fermeture ou la déformation de l'algèbre n'est encore autorisée**.


In [18]:

FINAL_STATUS = (
    "PARTIAL-PASS-HH-PREREQUISITE-ASSEMBLY_"
    "METRIC-MOMENTUM-LIFT-AND-LEGENDRE-DERIVATIVE-CLOSED_"
    "BLOCKED-CNLOC-COVARIANT-METRIC-FUNCTIONAL-JET"
)

assert (
    R_HH_STATUS
    ==
    "BLOCKED-MISSING-CNLOC-COVARIANT-METRIC-FUNCTIONAL-JET"
)

print("FINAL_STATUS =",FINAL_STATUS)
print("R_HH_STATUS =",R_HH_STATUS)
print("DISPERSION_READY =",DISPERSION_READY)


FINAL_STATUS = PARTIAL-PASS-HH-PREREQUISITE-ASSEMBLY_METRIC-MOMENTUM-LIFT-AND-LEGENDRE-DERIVATIVE-CLOSED_BLOCKED-CNLOC-COVARIANT-METRIC-FUNCTIONAL-JET
R_HH_STATUS = BLOCKED-MISSING-CNLOC-COVARIANT-METRIC-FUNCTIONAL-JET
DISPERSION_READY = False



# 18 — Export

L'artefact exporté doit conserver explicitement la raison du blocage pour empêcher qu'un notebook ultérieur transforme ce gate en fermeture implicite.


In [19]:

artifact = {
    "notebook":
        "GVH_Diagonal_Cubic_0.3.2.7.3.7.3.3.1.6",

    "traceability":
        "ACTUAL_GVH_UPSTREAM_NO_FORCED_CLOSURE",

    "target":
        {
            "RHH_canonical":
                "{H[N],H[M]}_can - D[beta]",

            "RHH_Dirac":
                "{H[N],H[M]}_D - D[beta]",

            "beta":
                "h^{ij}(N D_j M - M D_j N)",
        },

    "closed_inputs":
        {
            "Ci_exact":
                True,

            "QJU":
                True,

            "R3_functional_derivative":
                True,

            "DivB_all_pairs":
                True,

            "auxiliary_basis":
                [
                    "p_lambda",
                    "chi",
                    "psi",
                    "rho",
                ],

            "C4_rank_generic":
                4,

            "Dirac_inverse":
                True,

            "metric_collective_to_pi_density_map":
                {
                    "P_ii":
                        "2*pi^ii/sqrt(h)",
                    "P_ij_i_lt_j":
                        "4*pi^ij/sqrt(h)",
                },

            "CNloc_Legendre_derivative_identity":
                True,
        },

    "new_blocker":
        {
            "name":
                "CNloc_covariant_metric_functional_jet",

            "reason":
                (
                    "Local orthonormal Q,J,U depends on D_i v_j "
                    "and metric contractions but does not explicitly "
                    "encode h_ij/Gamma^k_ij variation required for "
                    "delta int N C_N_loc / delta h_ij."
                ),

            "required_identity":
                (
                    "delta(D_i v_j) = D_i(delta v_j) "
                    "- deltaGamma^k_ij v_k"
                ),
        },

    "full_HH_canonical_bracket_computed":
        False,

    "full_HH_Dirac_bracket_computed":
        False,

    "RHH_physical_classified":
        False,

    "RHH_status":
        R_HH_STATUS,

    "hypersurface_algebra_closed":
        False,

    "dispersion_ready":
        False,

    "next":
        (
            "GVH_Diagonal_Cubic_0.3.2.7.3.7.3.3.1.6.1_"
            "Covariant_CNloc_Metric_Functional_Jet_and_"
            "Connection_Variation_Closure.ipynb"
        ),

    "gates":
        GATES,

    "final_status":
        FINAL_STATUS,
}

export_dir = (
    Path("/content/gvh_exports")
    if Path("/content").exists()
    else Path.cwd()/"gvh_exports"
)

export_dir.mkdir(
    parents=True,
    exist_ok=True
)

artifact_path = export_dir / (
    "gvh_0.3.2.7.3.7.3.3.1.6_"
    "strict_HH_sufficiency_gate.json"
)

artifact_path.write_text(
    json.dumps(
        artifact,
        indent=2
    ),
    encoding="utf-8"
)

print("Artifact:",artifact_path)


Artifact: /content/gvh_exports/gvh_0.3.2.7.3.7.3.3.1.6_strict_HH_sufficiency_gate.json



# Conclusion

Les trois blocages historiquement identifiés avant le HH ont bien été considérablement réduits :

\[
R^{(3)}:
\quad \text{fermé},
\]

\[
D_iB^i:
\quad \text{fermé sur toutes les paires canoniques},
\]

\[
\{\Phi_A\},\,C_4^{-1},\,\text{réduction auxiliaire}:
\quad \text{prêts}.
\]

L'assemblage strict de `.1.6` révèle néanmoins une distinction qui ne peut pas être ignorée :

\[
\boxed{
\text{forme locale orthonormée de }C_N^{\rm loc}
\neq
\text{opérateur métrique fonctionnel covariant complet}.
}
\]

Parce que :

\[
D_iv_j
=
\partial_iv_j-\Gamma^k{}_{ij}v_k,
\]

le vrai calcul de :

\[
\frac{\delta H}{\delta h_{ij}}
\]

doit contenir la variation de connexion.

Tant que ce bloc n'est pas explicitement dérivé :

\[
\boxed{
R_{HH}
=
\texttt{NOT-YET-MATERIALIZED}
}
\]

\[
\boxed{
\texttt{HYPERSURFACE\_ALGEBRA\_CLOSED=False}
}
\]

\[
\boxed{
\mathrm{DISPERSION\_READY=False}.
}
\]

La prochaine opération autorisée est donc strictement ciblée sur ce dernier jet métrique covariant, sans modifier le modèle.
